<a href="https://colab.research.google.com/github/kuds/reinforce-tactics/blob/main/notebooks/ppo_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# Force headless pygame BEFORE anything imports pygame (directly or
# transitively). SDL picks its video driver at import time, so
# setting this later is a no-op for an already-loaded SDL backend.
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")

# Reinforce Tactics — PPO Training

This notebook trains a **MaskablePPO** agent against the **`RandomBot`**
opponent on the 6×6 beginner map. It uses an SB3 `EvalCallback`-style
periodic-eval pattern: a single `model.learn()` call drives `TOTAL_TIMESTEPS`
of training, and the eval callback runs `N_EVAL_EPISODES` evaluation
episodes every `EVAL_FREQ` env steps. Defaults: 5,000,000 total steps,
30 episodes every 100,000 steps (= 50 evaluations across the run).

We use the random opponent as the **starting curriculum**: it is weak enough
that the agent can experience early wins (and the +win terminal reward)
within only a few thousand steps. Once the agent reliably beats `RandomBot`,
graduate to the harder `SimpleBot` opponent by setting `OPPONENT = "simple"`
in the config cell.

At each evaluation the agent is scored on:
- **Win rate** (% of games won against the configured opponent)
- **Average episode reward**
- **Average episode length** (steps)

The best-so-far model (by eval win rate) is saved to
`BENCHMARK_DIR / "best_model.zip"` automatically, and the final model is
saved to `final_model.zip`.

**Runtime:** GPU recommended (L4 or better). CPU is possible but slower.

---

### Why MaskablePPO?

The game has a `MultiDiscrete` action space where many action combinations
are invalid at any given time (e.g. you can't attack a tile with no enemy).
**Action masking** prevents the agent from sampling these invalid actions,
which typically yields 2–3× faster convergence compared to plain PPO.


## 1. Setup

In [ ]:
# Install dependencies
!pip install -q gymnasium stable-baselines3 sb3-contrib tensorboard pandas numpy torch matplotlib opencv-python-headless

import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"  GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Clone repo and install as a package
import os
import sys
from pathlib import Path

REPO_DIR = Path("reinforce-tactics")
if REPO_DIR.exists():
    os.chdir(REPO_DIR)
elif Path("notebooks").exists():
    # Already inside the repo
    os.chdir("..")
else:
    print("Cloning repository...")
    !git clone https://github.com/kuds/reinforce-tactics.git
    os.chdir(REPO_DIR)

# Install the package so all imports resolve
!pip install -q -e .

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f"Working directory: {os.getcwd()}")

## 2. Imports

In [ ]:
import json
import time
from datetime import datetime

import matplotlib.pyplot as plt
import pandas as pd
from sb3_contrib import MaskablePPO

from reinforcetactics.rl.masking import make_maskable_env, make_maskable_vec_env

print("All imports successful.")

## 2b. Storage configuration

Choose where to save benchmark outputs. Set `USE_GOOGLE_DRIVE = True` to
persist results to Google Drive (recommended for Colab -- files survive
runtime disconnects). Set `False` to use the default local/Colab storage.

Each execution saves outputs under a **per-opponent, datetime-stamped subfolder**
(e.g. `benchmarks/ppo_vs_random/20250615_143022/`), so results are organized
by opponent and previous runs are preserved automatically.

In [ ]:
# --- Storage configuration ---
# Set to True to save all outputs (checkpoints, results, plots) to Google Drive.
# This is recommended when running on Google Colab, since files on the local
# Colab filesystem are lost when the runtime disconnects.
# Set to False to use the default local/Colab storage.
USE_GOOGLE_DRIVE = True

# Google Drive base directory (only used when USE_GOOGLE_DRIVE is True).
# This path is relative to your Google Drive root (MyDrive/).
# The final output path is: {DRIVE_BASE_DIR}/ppo_vs_{OPPONENT}/{datetime}/
DRIVE_BASE_DIR = "reinforce-tactics/benchmarks"

# --- Mount Google Drive if enabled ---
if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive

        drive.mount("/content/drive")
        SAVE_DIR = Path("/content/drive/MyDrive") / DRIVE_BASE_DIR
        SAVE_DIR.mkdir(parents=True, exist_ok=True)
        print("Google Drive mounted successfully.")
        print(f"Save directory: {SAVE_DIR}")
    except ImportError:
        print("WARNING: google.colab not available (not running in Colab).")
        print("  Falling back to local storage.")
        USE_GOOGLE_DRIVE = False
        SAVE_DIR = None
    except Exception as e:
        print(f"WARNING: Failed to mount Google Drive: {e}")
        print("  Falling back to local storage.")
        USE_GOOGLE_DRIVE = False
        SAVE_DIR = None
else:
    SAVE_DIR = None
    print("Using local storage (default).")
    print("  Tip: Set USE_GOOGLE_DRIVE = True to persist results to Google Drive.")

## 3. Configuration

In [ ]:
# --- Benchmark settings ---
MAP_FILE = "maps/1v1/starter.csv"  # 4x4 playable area inside 6x6 ocean borders
OPPONENT = "random"  # Start with random for easier wins
MAX_STEPS = 400  # max env steps per episode (raised so MAX_TURNS triggers first)
MAX_TURNS = 20  # max game turns before auto-draw (terminated, not truncated)
N_ENVS = 4  # parallel training envs
SEED = 42

# Enabled unit subset — keeps the action space smaller so the pipeline can be
# validated before we open up the full roster. W = Warrior (200g, 15 HP, mvmt 3),
# M = Mage (300g, 10 HP, mvmt 2, ranged), A = Archer (250g, 15 HP, mvmt 3, ranged).
# Both players (agent + RandomBot) are restricted via game_state.enabled_units;
# legal_actions filters create_unit accordingly (see bot.py:255).
ENABLED_UNITS = ["W", "M", "A"]

# Action space mode:
#   'flat_discrete'  — exact per-action masks (recommended, eliminates invalid actions)
#   'multi_discrete' — per-dimension masks (over-approximation, original behaviour)
ACTION_SPACE = "flat_discrete"

# Training & evaluation schedule (SB3 EvalCallback-style: total timesteps
# + a fixed eval frequency, replacing the older hand-picked checkpoint list).
TOTAL_TIMESTEPS = 5_000_000
EVAL_FREQ = 100_000  # run an evaluation every N env steps (across all envs)
N_EVAL_EPISODES = 30  # episodes per evaluation

# --- Reward configuration (Direction A) ---
# The agent must SEE that capturing the enemy HQ dominates farming.
# Strategy: heavy potential-based shaping (policy-invariant; rewards
# structural advantage like HQ control without being farmable) plus
# minimal per-action rewards. Per-action rewards bias the policy and
# previously created a stalling local optimum (~+400/ep farmed,
# capture worth less than continuing to farm). Now:
#   - Win/loss/draw all dominate any per-step shaping
#   - Draw == loss (timing out IS losing)
#   - structure_control in the potential function makes capturing the
#     HQ a large monotonic shaping bump that points the gradient toward
#     seizure without polluting the optimal policy.
REWARD_CONFIG = {
    # Terminal rewards — must dominate any farm strategy
    "win": 5000.0,
    "loss": -5000.0,
    "draw": -5000.0,
    # Potential-based shaping (policy-invariant; only Phi(s')-Phi(s) enters)
    "income_diff": 0.5,
    "unit_diff": 1.0,
    "structure_control": 10.0,
    # Per-action — minimal; only nudge toward HQ pressure
    "create_unit": 0.0,
    "move": 0.0,
    "damage_scale": 0.0,
    "kill": 0.0,
    "seize_progress": 10.0,
    "capture": 1000.0,
    "cure": 0.0,
    "heal_scale": 0.0,
    "paralyze": 0.0,
    "haste": 0.0,
    "defence_buff": 0.0,
    "attack_buff": 0.0,
    # Penalties
    "invalid_action": -10.0,
    "turn_penalty": -20.0,
}

# PPO hyperparameters
PPO_CONFIG = dict(
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    # Lowered from 0.05 — with the sharper terminal-dominated reward,
    # the policy needs to be allowed to concentrate. The previous run
    # had approx_kl pinned ~0.001 (way below the ~0.02 target), the
    # signature of an entropy bonus drowning the policy gradient. 0.01
    # keeps some exploration while letting the agent commit once it
    # finds a winning sequence.
    ent_coef=0.05,
    vf_coef=0.5,
    max_grad_norm=0.5,
)

# --- Per-run output directory (opponent + datetime-stamped) ---
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_SUBDIR = f"ppo_vs_{OPPONENT}"

# Output paths — uses Google Drive if enabled, otherwise local storage.
# Each execution gets its own subfolder: ppo_vs_{OPPONENT}/{datetime}/
if SAVE_DIR is not None:
    BENCHMARK_DIR = SAVE_DIR / RUN_SUBDIR / RUN_ID
else:
    BENCHMARK_DIR = Path("benchmarks") / RUN_SUBDIR / RUN_ID
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR = BENCHMARK_DIR / "charts"
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Map:          {MAP_FILE}")
print(f"Opponent:     {OPPONENT}")
print(f"Enabled units:{ENABLED_UNITS}")
print(f"Action space: {ACTION_SPACE}")
print(f"Max steps:    {MAX_STEPS}")
print(f"Max turns:    {MAX_TURNS}")
print(f"Total steps:  {TOTAL_TIMESTEPS:,}")
print(f"Eval freq:    {EVAL_FREQ:,} env steps  ({TOTAL_TIMESTEPS // EVAL_FREQ} evaluations)")
print(f"Eval eps:     {N_EVAL_EPISODES}")
print(f"ent_coef:     {PPO_CONFIG['ent_coef']}")
print(f"turn_penalty: {REWARD_CONFIG['turn_penalty']}")
print(f"draw:         {REWARD_CONFIG['draw']}")
print(f"Storage:      {'Google Drive' if USE_GOOGLE_DRIVE else 'Local'}")
print(f"Run ID:       {RUN_ID}")
print(f"Output dir:   {BENCHMARK_DIR}")

## 4. Create environments

In [ ]:
# Training envs (vectorized, headless)
vec_env = make_maskable_vec_env(
    n_envs=N_ENVS,
    map_file=MAP_FILE,
    opponent=OPPONENT,
    max_steps=MAX_STEPS,
    max_turns=MAX_TURNS,
    reward_config=REWARD_CONFIG,
    enabled_units=ENABLED_UNITS,
    seed=SEED,
    use_subprocess=False,  # DummyVecEnv (safer in notebooks)
    action_space_type=ACTION_SPACE,
)

# Single eval env, deterministic given SEED + episode index. The eval callback
# below feeds it through evaluate_model() at fixed timestep intervals.
eval_env = make_maskable_env(
    map_file=MAP_FILE,
    opponent=OPPONENT,
    max_steps=MAX_STEPS,
    max_turns=MAX_TURNS,
    reward_config=REWARD_CONFIG,
    enabled_units=ENABLED_UNITS,
    action_space_type=ACTION_SPACE,
    seed=SEED,
)

print(f"Observation space: {vec_env.observation_space}")
print(f"Action space:      {vec_env.action_space}")

## 5. Create MaskablePPO model

In [ ]:
model = MaskablePPO(
    "MultiInputPolicy",
    vec_env,
    verbose=0,
    tensorboard_log=str(BENCHMARK_DIR / "tensorboard"),
    device=DEVICE,
    seed=SEED,
    **PPO_CONFIG,
)

print("MaskablePPO model created.")
print(f"Policy:  {model.policy.__class__.__name__}")
print(f"Device:  {model.device}")

## 6. Library imports

Pull in the project's evaluation helper, the SB3 callback subclasses
(`TrainingMetricsCallback`, `PeriodicEvalCallback`), and the plot
helpers from `reinforcetactics.rl`. Keeping these in the library —
rather than inline in this notebook — means other notebooks and the
`scripts/train/train_*.py` scripts can reuse the same primitives.


In [ ]:
from reinforcetactics.rl.callbacks import (
    PeriodicEvalCallback,
    TrainingMetricsCallback,
)
from reinforcetactics.rl.viz import (
    plot_action_distribution,
    plot_eval_curves,
    plot_outcome_breakdown,
    plot_reward_decomposition,
)

print("Imported library helpers (evaluation, callbacks, viz).")

In [ ]:
metrics_callback = TrainingMetricsCallback()
eval_callback = PeriodicEvalCallback(
    eval_env,
    eval_freq=EVAL_FREQ,
    n_eval_episodes=N_EVAL_EPISODES,
    eval_seed_base=SEED,
    save_dir=BENCHMARK_DIR,
    track_breakdown=True,  # capture action counts + reward components per eval
)
print("TrainingMetricsCallback + PeriodicEvalCallback ready.")

## 7. Train and evaluate

A single ``model.learn(total_timesteps=TOTAL_TIMESTEPS, ...)`` call drives
the run. ``TrainingMetricsCallback`` captures per-rollout PPO internals
(``train/approx_kl``, ``train/explained_variance``, etc.); ``PeriodicEvalCallback``
runs ``evaluate_model`` every ``EVAL_FREQ`` env steps and accumulates results
plus per-step action / reward-component breakdowns into ``eval_callback.results``.
The best-by-win-rate model is saved automatically.


In [ ]:
# Single model.learn() call. The eval callback fires every EVAL_FREQ env
# steps and appends an entry to ``eval_callback.results`` — that list is what
# the downstream plots, the diagnose_training cell, and the saved JSON all
# read from (results = eval_callback.results below).
print(f"Training for {TOTAL_TIMESTEPS:,} timesteps ({TOTAL_TIMESTEPS // EVAL_FREQ} evaluations every {EVAL_FREQ:,} steps)...")

start_time = time.time()
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    reset_num_timesteps=False,
    progress_bar=True,
    callback=[metrics_callback, eval_callback],
)
total_time = time.time() - start_time

results = eval_callback.results
model.save(str(BENCHMARK_DIR / "final_model.zip"))

print(f"\nTotal wall time: {total_time / 60:.1f} minutes")
print(f"Captured {len(results)} eval snapshots and {len(metrics_callback.records)} training-metric snapshots.")
print(
    f"Best win rate during training: {eval_callback.best_win_rate * 100:.1f}%  (saved to {BENCHMARK_DIR / 'best_model.zip'})"
)

## 8. Results table

In [ ]:
df = pd.DataFrame(results)
df["win_rate_pct"] = (df["win_rate"] * 100).round(1)
df["avg_reward"] = df["avg_reward"].round(1)
df["avg_length"] = df["avg_length"].round(1)

display_df = df[["timesteps", "win_rate_pct", "avg_reward", "avg_length", "wins", "losses", "draws"]].copy()
display_df.columns = ["Timesteps", "Win Rate (%)", "Avg Reward", "Avg Length", "Wins", "Losses", "Draws"]
display_df = display_df.set_index("Timesteps")

# With many evals (default schedule = 50) the table is short enough to show
# in full; if EVAL_FREQ is reduced, fall back to head + tail so the table
# stays readable. The full table is always in the saved CSV.
MAX_TABLE_ROWS = 30
if len(display_df) > MAX_TABLE_ROWS:
    half = MAX_TABLE_ROWS // 2
    print(f"Showing first {half} and last {half} of {len(display_df)} eval rows. Full table is in the saved CSV.")
    display_df = pd.concat([display_df.head(half), display_df.tail(half)])
display_df

## 9. Training curves

Win rate, average reward, episode length over time, plus two PPO update
health diagnostics (`approx_kl` and `explained_variance`). If `approx_kl`
is pinned near zero the policy isn't actually updating; if
`explained_variance` is near zero the value head isn't fitting the
returns. Either typically indicates the agent is **not** learning,
regardless of what the reward curve looks like.


In [ ]:
# Eval curves + PPO update health (approx_kl, explained_variance) on a
# single shared-axis figure. The helper handles the case where train_records
# is empty (e.g. resumed run with metrics_callback.records cleared).
fig = plot_eval_curves(
    results,
    train_records=metrics_callback.records,
    opponent_label=OPPONENT,
    charts_dir=CHARTS_DIR,
)
plt.show()

## 9a. Reward decomposition over time

Stacked area of the four reward components — `action`, `shaping_delta`,
`invalid_penalty`, `terminal` — summed across all eval episodes at each
evaluation point. Reads from the `track_breakdown=True` data captured by
the eval callback.

**What the shapes mean:**
- A growing **orange** band (`terminal`) means the agent is winning more
  decisive games. This is what you want.
- A dominant **green** band (`action`) with `terminal` ≈ 0 means the
  agent farms per-action shaping rewards but never converts wins —
  classic shaping-reward stalemate.
- A large **grey** band (`invalid_penalty`) means action masking isn't
  doing its job; check `ACTION_SPACE = "flat_discrete"`.
- The **blue** band (`shaping_delta`) is potential-based reward shaping;
  it should oscillate around zero, not consistently dominate.


In [ ]:
fig = plot_reward_decomposition(results, charts_dir=CHARTS_DIR)
if fig is not None:
    plt.show()

## 9b. Action distribution over time

Stacked area of action-type usage % per eval. Each column normalizes to
100% so this answers "**which** actions does the agent prefer?" rather
than "how many steps did it take?".

**What the shapes mean:**
- Healthy: blue (`create_unit`) early, then green (`move`) and orange
  (`seize`) growing as the agent learns to capture structures.
- Collapsed: a dominant grey (`end_turn`) band — the agent has learned
  that ending its turn avoids penalties (especially common when
  `REWARD_CONFIG["invalid_action"]` is large in magnitude).
- Stuck: distribution is flat over time — the policy isn't differentiating
  between situations, often correlated with `approx_kl ≈ 0` above.


In [ ]:
fig = plot_action_distribution(results, charts_dir=CHARTS_DIR)
if fig is not None:
    plt.show()

## 9c. Outcome × end-reason breakdown

Splits each eval's win/loss/draw counts by *how* the game ended:

- **`hq_capture`** — the intended win condition. The seizing player
  captured the enemy headquarters.
- **`elimination`** — every unit on one side died. With respawning
  buildings this is rare in long games; usually means whoever it was
  spent the whole game starved of structures.
- **`max_turns_draw`** — game hit `MAX_TURNS` with no decisive event.
- **`max_steps_truncate`** — env step counter ran out before the game
  itself terminated. Should be near-zero if `MAX_STEPS` is set above
  `MAX_TURNS × ~17 actions/turn`.

A 95% win rate where wins are 90% `hq_capture` is a healthy goal-directed
agent. The same 95% where wins are mostly `elimination` is a different
strategy — useful but not what the reward function was designed for.


In [ ]:
fig = plot_outcome_breakdown(results, charts_dir=CHARTS_DIR)
if fig is not None:
    plt.show()

## 9d. Watch the Agent Play (Video Replay)

Record the trained agent playing a full game and view it as an inline
MP4 video. This uses **headless rendering** via `pygame-ce` — no display
server required (works on Colab).

The video shows the actual game map with terrain, structures, units,
health bars, and player colours exactly as they appear in the game UI.

In [ ]:
# pygame-ce is required for headless rendering. SDL_VIDEODRIVER=dummy was
# already set in the very first cell of the notebook; setting it again here
# would be a no-op once SDL has picked a backend.
try:
    import pygame

    print(f"pygame version: {pygame.ver}")
except ImportError:
    print("Installing pygame-ce...")
    !pip install -q pygame-ce
    import pygame

    print(f"pygame version: {pygame.ver}")

from reinforcetactics.utils.video import display_video_in_notebook, record_evaluation_to_video

print("Video recording utilities loaded.")

In [ ]:
# Sanity check before recording. If sprites still load with
# USE_PIXEL_ART=False below, the kernel is running stale code —
# Runtime > Disconnect and delete runtime, then re-run from the top.
import os

import pygame

from reinforcetactics.ui.renderer import Renderer

print("SDL_VIDEODRIVER:", os.environ.get("SDL_VIDEODRIVER"))
print("pygame display init:", pygame.display.get_init())

_probe_env = make_maskable_env(map_file=MAP_FILE, opponent=OPPONENT, enabled_units=ENABLED_UNITS)
_probe_env.reset()
_r = Renderer(_probe_env.unwrapped.game_state, replay_mode=True, headless=True, pixel_art=False)
print("screen:", type(_r.screen).__name__, _r.screen.get_size())
print("tile sprites loaded:", sum(v is not None for v in _r.tile_images.values()))
print("unit sprites loaded:", len(_r.unit_images))
print("animator:", _r.animator)
_probe_env.close()

In [ ]:
# Record one full game of the trained agent
# This creates a fresh environment (not the eval_env used above) to avoid
# state leakage, then runs the model through a complete episode while
# capturing every frame with the headless renderer.
#
# We record TWO videos:
#   - agent_replay_best.mp4  — best-by-eval-win-rate checkpoint
#                              (loaded from BENCHMARK_DIR / "best_model.zip")
#   - agent_replay_final.mp4 — last-step model (the in-memory `model`,
#                              also saved to "final_model.zip")
# These can diverge meaningfully once the policy oscillates; comparing
# them is a quick sanity check on whether eval is picking a real peak.
#
# A standard reinforce-tactics replay JSON is also written next to each
# MP4 (agent_replay_best.json / agent_replay_final.json) via
# GameState.save_replay_to_file. These are loadable with FileIO.load_replay
# and replayable through record_replay_to_video / the in-game replay viewer.

# Set to True to render with the bundled pixel-art sprites
# (assets/sprites/) instead of the default coloured rects + unit letters.
USE_PIXEL_ART = False


def _record(_model, suffix, label):
    _env = make_maskable_env(
        map_file=MAP_FILE,
        opponent=OPPONENT,
        max_steps=MAX_STEPS,
        max_turns=MAX_TURNS,
        reward_config=REWARD_CONFIG,
        enabled_units=ENABLED_UNITS,
        action_space_type=ACTION_SPACE,
    )
    _path = str(BENCHMARK_DIR / f"agent_replay_{suffix}.mp4")
    print(f"Recording {label} agent gameplay -> {_path}")
    _result = record_evaluation_to_video(
        env=_env,
        model=_model,
        output_path=_path,
        fps=4,
        max_steps=MAX_STEPS,
        deterministic=True,
        use_pixel_art=USE_PIXEL_ART,
    )
    _env.close()
    winner_str = {1: "Agent wins", 2: "Opponent wins", None: "Draw"}
    print(
        f"  result={winner_str.get(_result['winner'], 'Unknown')}  reward={_result['total_reward']:.1f}  steps={_result['steps']}"
    )
    if _result.get("replay_path"):
        print(f"  replay JSON: {_result['replay_path']}")
    return _result


# Best checkpoint (saved by PeriodicEvalCallback).
best_path = BENCHMARK_DIR / "best_model.zip"
if best_path.exists():
    best_model = MaskablePPO.load(str(best_path), device=DEVICE)
    result_best = _record(best_model, "best", "BEST")
else:
    print(f"No best_model.zip at {best_path} — skipping BEST recording (likely 0 wins ever).")
    result_best = None

# Final model — the in-memory `model` after model.learn() finished.
result_final = _record(model, "final", "FINAL")

# `result` is what the original notebook flow expected; keep it pointing
# at the better of the two (best if it exists, else final).
result = result_best if result_best is not None else result_final

In [ ]:
# Display the recorded video(s) inline in the notebook.
# On Colab this embeds the MP4 directly; locally it opens in the browser.
if result_best is not None:
    print("BEST checkpoint:")
    display_video_in_notebook(result_best["video_path"])
print("FINAL checkpoint:")
display_video_in_notebook(result_final["video_path"])

### 9e. Individual game statistics (2x3)

Per-step diagnostics for the recorded game. Useful for asking "what
did the agent *actually* do this game?" — independent of the
aggregate eval curves above.

- **Top row (per player over time):** unit count, gold, structures owned (towers + buildings + HQ)
- **Bottom-left:** action mix used in this game — `create_unit` is split
  by which type was actually spawned (`create_W`, `create_M`, …) so you
  can see e.g. "the agent only ever built Warriors". Blue bars = creates,
  green bars = other actions.
- **Bottom-middle:** cumulative reward by component (`action`,
  `shaping_delta`, `invalid_penalty`, `terminal`)
- **Bottom-right:** outcome summary — winner, end reason
  (`hq_capture` / `elimination` / `max_turns_draw` / `max_steps_truncate`),
  final unit/gold/structure counts, and per-type creation counts.

Reads from `result["step_stats"]` populated by
`record_evaluation_to_video`. Plots both BEST (if available) and FINAL
checkpoints.


In [ ]:
# 2x3 per-step stats for the recorded game.
import numpy as np

ACTION_NAMES = {
    0: "create_unit",
    1: "move",
    2: "attack",
    3: "seize",
    4: "heal",
    5: "end_turn",
    6: "paralyze",
    7: "haste",
    8: "defence_buff",
    9: "attack_buff",
}


def plot_individual_game_stats(result_dict, title_suffix=""):
    step_stats = result_dict.get("step_stats") or []
    if not step_stats:
        print("No step_stats on result — re-run the recording cell with the updated record_evaluation_to_video.")
        return None

    steps = np.arange(len(step_stats))
    agent_units = [s["agent_units"] for s in step_stats]
    opp_units = [s["opponent_units"] for s in step_stats]
    agent_gold = [s["agent_gold"] for s in step_stats]
    opp_gold = [s["opponent_gold"] for s in step_stats]
    agent_struct = [s.get("agent_structures", 0) for s in step_stats]
    opp_struct = [s.get("opponent_structures", 0) for s in step_stats]

    # Action counts (skip the initial snapshot which has action_type=None).
    # Break create_unit out by which unit type was actually spawned, so the
    # bar shows e.g. create_W / create_M / create_A separately. Other action
    # types stay aggregated.
    action_counts: dict[str, int] = {}
    for s in step_stats:
        at = s.get("action_type")
        if at is None:
            continue
        if int(at) == 0:
            ut = s.get("unit_type")
            name = f"create_{ut}" if ut else "create_unit"
        else:
            name = ACTION_NAMES.get(int(at), str(at))
        action_counts[name] = action_counts.get(name, 0) + 1

    # Cumulative reward components
    components = ["action", "shaping_delta", "invalid_penalty", "terminal"]
    cum = {c: np.zeros(len(step_stats)) for c in components}
    running = {c: 0.0 for c in components}
    for i, s in enumerate(step_stats):
        rb = s.get("reward_breakdown") or {}
        for c in components:
            running[c] += float(rb.get(c, 0.0) or 0.0)
            cum[c][i] = running[c]

    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    suffix = f" — {title_suffix}" if title_suffix else ""
    fig.suptitle(f"Individual game statistics{suffix}", fontsize=13, fontweight="bold")

    ax = axes[0, 0]
    ax.plot(steps, agent_units, label="agent", color="#1f77b4")
    ax.plot(steps, opp_units, label="opponent", color="#d62728")
    ax.set_title("Unit count")
    ax.set_xlabel("Step")
    ax.set_ylabel("Units")
    ax.legend()
    ax.grid(alpha=0.3)

    ax = axes[0, 1]
    ax.plot(steps, agent_gold, label="agent", color="#1f77b4")
    ax.plot(steps, opp_gold, label="opponent", color="#d62728")
    ax.set_title("Gold")
    ax.set_xlabel("Step")
    ax.set_ylabel("Gold")
    ax.legend()
    ax.grid(alpha=0.3)

    ax = axes[0, 2]
    ax.plot(steps, agent_struct, label="agent", color="#1f77b4")
    ax.plot(steps, opp_struct, label="opponent", color="#d62728")
    ax.set_title("Structures owned (towers + buildings + HQ)")
    ax.set_xlabel("Step")
    ax.set_ylabel("Structures")
    ax.legend()
    ax.grid(alpha=0.3)

    ax = axes[1, 0]
    if action_counts:
        names = list(action_counts.keys())
        counts = [action_counts[n] for n in names]
        order = np.argsort(counts)[::-1]
        names = [names[i] for i in order]
        counts = [counts[i] for i in order]
        # Color creates differently from non-create actions so the unit-type
        # mix pops out of the bar.
        bar_colors = ["#1f77b4" if n.startswith("create_") else "#2ca02c" for n in names]
        ax.bar(names, counts, color=bar_colors)
        ax.set_title(f"Action mix (n={sum(counts)}, blue=creates)")
        ax.set_ylabel("Count")
        ax.tick_params(axis="x", rotation=45)
    else:
        ax.text(0.5, 0.5, "No actions recorded", ha="center", va="center")
        ax.set_axis_off()
    ax.grid(alpha=0.3, axis="y")

    ax = axes[1, 1]
    colors = {
        "action": "#2ca02c",
        "shaping_delta": "#1f77b4",
        "invalid_penalty": "#7f7f7f",
        "terminal": "#ff7f0e",
    }
    for c in components:
        ax.plot(steps, cum[c], label=c, color=colors[c])
    ax.set_title(f"Cumulative reward by component (total={result_dict.get('total_reward', 0):.0f})")
    ax.set_xlabel("Step")
    ax.set_ylabel("Cumulative reward")
    ax.legend(loc="best", fontsize=8)
    ax.grid(alpha=0.3)
    ax.axhline(0, color="k", lw=0.5)

    # Outcome summary panel — pulls from the result dict so the chart
    # tells you immediately whether the recorded game was an HQ capture,
    # an elimination, or a timeout, without scrolling back to the print.
    # Also lists per-unit-type creation counts so you can see e.g. "agent
    # only ever spawned Warriors" at a glance.
    ax = axes[1, 2]
    ax.set_axis_off()
    winner_str = {1: "Agent wins", 2: "Opponent wins", None: "Draw"}.get(result_dict.get("winner"), "Unknown")
    end_reason = result_dict.get("end_reason") or "n/a"
    final_step = step_stats[-1] if step_stats else {}
    create_lines = [f"  {n}: {c}" for n, c in sorted(action_counts.items()) if n.startswith("create_")]
    summary = [
        f"Result:        {winner_str}",
        f"End reason:    {end_reason}",
        f"Steps:         {result_dict.get('steps', 0)}",
        f"Final turn:    {final_step.get('turn', 'n/a')}",
        f"Total reward:  {result_dict.get('total_reward', 0):.0f}",
        "",
        f"Final units:   {final_step.get('agent_units', 0)} vs {final_step.get('opponent_units', 0)}",
        f"Final gold:    {final_step.get('agent_gold', 0)} vs {final_step.get('opponent_gold', 0)}",
        f"Final struct:  {final_step.get('agent_structures', 0)} vs {final_step.get('opponent_structures', 0)}",
        "",
        "Units created (agent):",
        *(create_lines or ["  (none)"]),
    ]
    ax.text(0.0, 0.95, "\n".join(summary), ha="left", va="top", family="monospace", fontsize=9, transform=ax.transAxes)
    ax.set_title("Outcome summary")

    fig.tight_layout(rect=[0, 0, 1, 0.96])
    out_path = CHARTS_DIR / f"individual_game_stats{('_' + title_suffix) if title_suffix else ''}.png"
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    print(f"Saved: {out_path}")
    return fig


if result_best is not None:
    plot_individual_game_stats(result_best, title_suffix="best")
    plt.show()
plot_individual_game_stats(result_final, title_suffix="final")
plt.show()

## 10. Save results

In [ ]:
# Save benchmark results as JSON
import subprocess


def _git_commit_id() -> str:
    """Return the current HEAD commit (short hash + dirty flag), or 'unknown'."""
    try:
        sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL, text=True).strip()
        dirty = subprocess.call(["git", "diff", "--quiet", "HEAD"], stderr=subprocess.DEVNULL)
        return f"{sha}-dirty" if dirty else sha
    except (subprocess.CalledProcessError, FileNotFoundError):
        return "unknown"


benchmark_data = {
    "metadata": {
        "date": datetime.now().isoformat(),
        "git_commit": _git_commit_id(),
        "map": MAP_FILE,
        "opponent": OPPONENT,
        "enabled_units": ENABLED_UNITS,
        "max_steps": MAX_STEPS,
        "max_turns": MAX_TURNS,
        "n_envs": N_ENVS,
        "total_timesteps": TOTAL_TIMESTEPS,
        "eval_freq": EVAL_FREQ,
        "n_eval_episodes": N_EVAL_EPISODES,
        "seed": SEED,
        "device": DEVICE,
        "ppo_config": PPO_CONFIG,
        "reward_config": REWARD_CONFIG,
        "storage": "google_drive" if USE_GOOGLE_DRIVE else "local",
    },
    "results": results,
}

results_path = BENCHMARK_DIR / "benchmark_results.json"
with open(results_path, "w") as f:
    json.dump(benchmark_data, f, indent=2)

print(f"Saved results:  {results_path}")
print(f"Git commit:     {benchmark_data['metadata']['git_commit']}")

# Also save as CSV for easy viewing
csv_path = BENCHMARK_DIR / "benchmark_results.csv"
df.to_csv(csv_path, index=False)
print(f"Saved CSV:      {csv_path}")

# List all saved files
print("\nAll benchmark files:")
for p in sorted(BENCHMARK_DIR.rglob("*")):
    if p.is_file():
        size = p.stat().st_size
        if size > 1024 * 1024:
            size_str = f"{size / 1024 / 1024:.1f} MB"
        elif size > 1024:
            size_str = f"{size / 1024:.1f} KB"
        else:
            size_str = f"{size} B"
        rel = p.relative_to(BENCHMARK_DIR)
        print(f"  {str(rel):40s}  {size_str}")

if USE_GOOGLE_DRIVE:
    print("\nFiles are saved to Google Drive at:")
    print(f"  My Drive/{DRIVE_BASE_DIR}/{RUN_SUBDIR}/{RUN_ID}/")
    print("  These files will persist after the Colab runtime disconnects.")
else:
    print("\nFiles are saved to local storage at:")
    print(f"  {BENCHMARK_DIR.resolve()}")
    print("  WARNING: These files will be lost if the Colab runtime disconnects.")
    print("  Set USE_GOOGLE_DRIVE = True in the storage config cell to persist them.")

## 11. TensorBoard (optional)

Launch TensorBoard to inspect detailed training metrics (loss, entropy, etc.).

In [ ]:
# Uncomment to launch TensorBoard inline:
# %load_ext tensorboard
# %tensorboard --logdir benchmarks/ppo_vs_simplebot/tensorboard

print("To view TensorBoard locally, run:")
print(f"  tensorboard --logdir {BENCHMARK_DIR / 'tensorboard'}")

## 12. Interpreting the results

### What to expect (vs `RandomBot`)

`RandomBot` is intentionally weak (uniform sampling from legal actions),
so a well-trained agent should saturate quickly. With the default schedule
(5M timesteps / eval every 100K) you should see win rate climb from
~10–40% in the first few evals, cross 70% by ~200K-500K, and plateau
above ~90% well before the run completes. Once the curve plateaus, switch
`OPPONENT = "simple"` (`SimpleBot`) for a stiffer challenge — expect
lower starting win rates and a slower climb.

**Note:** Exact numbers depend on hardware and random seed. The important
thing is the curve has a similar *shape* — monotonically increasing win
rate with diminishing returns once the agent dominates.

### If your results differ significantly

- **Win rate stuck at 0%:** The agent has never experienced a +win
  reward. Check that the eval episode lengths aren't all hitting
  `MAX_STEPS` (visible in the training curve — flat top means timeouts).
  Consider lowering `MAX_STEPS`, making `REWARD_CONFIG["draw"]` more
  negative, or shrinking the per-action shaping rewards.
- **Win rate oscillates / declines:** Try reducing the learning rate or
  raising the batch size. Check `eval/win_rate` and `train/approx_kl`
  in TensorBoard — runaway KL means the policy is over-shooting updates.
- **Action-masking warning:** With `ACTION_SPACE = "flat_discrete"`
  the agent should rarely sample invalid actions; if it does, the masks
  are broken upstream.
- **Much better than expected:** You may have found better
  hyperparameters! Consider contributing them back.

### Next steps

1. **Graduate the opponent:** Set `OPPONENT = "simple"` once you've saturated `random`
2. **Try different maps:** Larger maps (10×10, 14×14) are harder
3. **Tune hyperparameters:** Adjust `ent_coef`, `learning_rate`, etc.
4. **Self-play training:** See `scripts/train/train_self_play.py`
5. **AlphaZero:** See `scripts/train/train_alphazero.py` for MCTS-based training


In [ ]:
# Clean up environments
vec_env.close()
eval_env.close()
print("Done.")

## 15. Disconnect the Colab Runtime

Free the Colab GPU after the run completes so we don't keep burning
compute quota while the artifacts upload to Google Drive.

In [ ]:
import time

print("Training finished. Disconnecting runtime in 5 seconds...")
time.sleep(5)

from google.colab import runtime

runtime.unassign()